# Entrega 3 · Análisis semántico — USPG

## 1. Introducción al análisis semántico

Objetivo: continuar el compilador existente comprobando nombres y tipos del AST.
No se ejecutan instrucciones ni se genera TAC. Este notebook importa los módulos
reales, demuestra `%` y ejecuta la suite completa con assertions.

Fuente: [plantilla del docente](https://github.com/ciarenasg-uspg/compilador-entrega-semantica/blob/5fc51c0a3de764118b43cf6fac5645f730dde6b5/semantic/analizador.py).
El README enumera seis métodos; el archivo contiene cinco TODO porque el visitante
binario ya está implementado. Se completaron cinco y se amplió el binario para `%`.

In [1]:
from pathlib import Path
import sys
raiz = Path.cwd().resolve()
if not (raiz / 'main.py').is_file():
    raiz = raiz.parent
assert (raiz / 'semantic/analizador.py').is_file(), 'Ejecutar desde la raíz o notebooks/'
sys.path.insert(0, str(raiz))
from main import compilar, validar
from lexer.lexer import Lexer
from semantic.analizador import AnalizadorSemantico
print('Python:', sys.version.split()[0])
print('Importaciones del proyecto completas.')

Python: 3.13.1
Importaciones del proyecto completas.


## 2. Errores léxicos, sintácticos y semánticos

El lexer rechaza caracteres desconocidos; el parser rechaza estructuras inválidas;
la semántica detecta incoherencias en programas con sintaxis válida. Capturamos
errores esperados y comprobamos su clase, sin ocultar excepciones inesperadas.

In [2]:
casos = [('léxico', 'entero x = @;', SyntaxError),
         ('sintáctico', 'entero x = ;', SyntaxError),
         ('semántico', 'imprimir desconocida;', NameError)]
for fase, fuente, esperado in casos:
    try:
        validar(fuente)
    except esperado as error:
        print(f'{fase}: {type(error).__name__}: {error}')
    else:
        raise AssertionError(f'No se detectó el error {fase}')

léxico: SyntaxError: Error léxico: carácter inesperado '@' en línea 1, columna 12
sintáctico: SyntaxError: Línea 1: se esperaba número, identificador o expresión entre paréntesis
semántico: NameError: [Error Semantico] Línea 1: variable 'desconocida' no declarada.


## 3. Arquitectura Lexer → Parser → AST → Semántica

`compilar` conserva el retorno histórico `(tokens, ast)` y `validar` agrega el
analizador semántico. Visitor decide el método según la clase de cada nodo.

In [3]:
tokens, ast, sem = validar('entero edad = 20; imprimir edad;')
print('Tipos de token:', [t.tipo for t in tokens])
print('AST:', ast)
print('Tabla:', sem.tabla_simbolos)
assert tokens[-1].tipo == 'EOF'
assert sem.tabla_simbolos == {'edad': 'entero'}

Tipos de token: ['PR_ENTERO', 'ID', 'OP_ASIG', 'NUMERO', 'PUNTO_COMA', 'PR_IMPRIMIR', 'ID', 'PUNTO_COMA', 'EOF']
AST: Programa(sentencias=[Declaracion(nombre='edad', valor=Numero(valor=20)), Imprimir(expresion=Identificador(nombre='edad'))])
Tabla: {'edad': 'entero'}


## 4. Concepto y funcionamiento de la tabla de símbolos

Diccionario global `{nombre: tipo}`. La presencia indica declaración. No guarda
valores, estado de inicialización ni ámbitos locales. Registramos después de
validar cada inicializador. Esta traza usa el ejemplo real del operador módulo.

In [4]:
fuente_modulo = (raiz / 'pruebas/07_token_personalizado.txt').read_text(encoding='utf-8')
_, arbol_modulo = compilar(fuente_modulo)
tabla = AnalizadorSemantico()
print('Inicial:', dict(tabla.tabla_simbolos))
for indice, sentencia in enumerate(arbol_modulo.sentencias, 1):
    tabla.analizar(sentencia)
    print(indice, type(sentencia).__name__, dict(tabla.tabla_simbolos))
assert tabla.tabla_simbolos == {'numero': 'entero', 'divisor': 'entero', 'residuo': 'entero'}

Inicial: {}
1 Declaracion {'numero': 'entero'}
2 Declaracion {'numero': 'entero', 'divisor': 'entero'}
3 Declaracion {'numero': 'entero', 'divisor': 'entero', 'residuo': 'entero'}
4 Si {'numero': 'entero', 'divisor': 'entero', 'residuo': 'entero'}


## 5. Declaraciones e identificadores

Una declaración rechaza duplicados y comprueba el inicializador antes de registrar.
Un identificador debe existir y devuelve el tipo registrado. Se conserva `entero x;`:
la plantilla no exige comprobar inicialización.

In [5]:
def verificar_error(fuente, clase):
    try:
        validar(fuente)
    except clase as error:
        print(type(error).__name__, str(error))
    else:
        raise AssertionError('Se esperaba ' + clase.__name__)

verificar_error('entero x = x + 1;', NameError)
verificar_error('entero x = 1; entero x = 2;', NameError)
assert validar('entero x; imprimir x;')[2].tabla_simbolos == {'x': 'entero'}

NameError [Error Semantico] Línea 1: variable 'x' no declarada.
NameError [Error Semantico] Línea 1: variable 'x' ya declarada.


## 6. Validación de tipos y operaciones

Los literales y variables tienen tipo entero. `+ - * / %` requieren enteros y
devuelven entero. `== != < > <= >=` requieren enteros y devuelven booleano.
No se declaran variables booleanas, pero las comparaciones permiten comprobar
errores de tipo representables. `imprimir` admite cualquier expresión válida.

In [6]:
for expresion in ('5 + 3 * 2', '10 / 3', '1 < 2', '3 >= 1'):
    expr = compilar('imprimir ' + expresion + ';')[1].sentencias[0].expresion
    print(expresion, '→', AnalizadorSemantico().analizar(expr))
verificar_error('entero x = 1 < 2;', TypeError)
verificar_error('entero x = 2; x = 1 < 2;', TypeError)
verificar_error('entero x = (1 < 2) + 3;', TypeError)

5 + 3 * 2 → entero
10 / 3 → entero
1 < 2 → booleano
3 >= 1 → booleano
TypeError [Error Semantico] Línea 1: inicialización de 'x' exige 'entero', se recibió 'booleano'.
TypeError [Error Semantico] Línea 1: asignación a 'x' exige 'entero', se recibió 'booleano'.
TypeError [Error Semantico] Operador '+' exige dos enteros, se recibieron 'booleano' y 'entero'.


## 7. Validación de condiciones y ciclos

Las condiciones exigen booleano. Se recorren los bloques una vez, sin evaluar
condiciones ni ejecutar ciclos. La tabla es global: se registran incluso las
declaraciones de bloques cuya condición aparenta ser falsa.

In [7]:
sem = validar('entero x = 0; mientras (x < 3) { x = x + 1; }')[2]
assert sem.tabla_simbolos == {'x': 'entero'}
print('Ciclo validado sin ejecutarse:', sem.tabla_simbolos)
verificar_error('si (3) {}', TypeError)
verificar_error('mientras (1) {}', TypeError)
verificar_error('si (1 > 2) { imprimir ausente; }', NameError)

Ciclo validado sin ejecutarse: {'x': 'entero'}
TypeError [Error Semantico] Condición de 'si' exige 'booleano', se recibió 'entero'.
TypeError [Error Semantico] Condición de 'mientras' exige 'booleano', se recibió 'entero'.
NameError [Error Semantico] Línea 1: variable 'ausente' no declarada.


## 8. Extensión personal: operador módulo

`%` conserva `OP_MODULO`, la precedencia de producto y el nodo `OpBinaria`.
Se amplía únicamente el conjunto aritmético semántico: entero × entero → entero.
No se calcula el residuo ni se comprueban divisores cero en esta entrega.

In [8]:
tokens, ast, sem = validar(fuente_modulo)
assert any(t.tipo == 'OP_MODULO' and t.valor == '%' for t in tokens)
assert ast.sentencias[2].valor.operador == '%'
print('Extensión preservada:', sem.tabla_simbolos)
verificar_error('entero residuo = (1 < 2) % 3;', TypeError)
verificar_error('entero residuo = 3 % (1 < 2);', TypeError)

Extensión preservada: {'numero': 'entero', 'divisor': 'entero', 'residuo': 'entero'}
TypeError [Error Semantico] Operador '%' exige dos enteros, se recibieron 'booleano' y 'entero'.
TypeError [Error Semantico] Operador '%' exige dos enteros, se recibieron 'entero' y 'booleano'.


## 9. Ejemplos válidos

Los cinco programas siguientes conservan la sintaxis de entregas anteriores.
Se comprueba todo el flujo sin calcular resultados.

In [9]:
for prefijo in ('01_', '02_', '03_', '07_', '13_'):
    archivo = next((raiz / 'pruebas').glob(prefijo + '*.txt'))
    _, _, resultado = validar(archivo.read_text(encoding='utf-8'))
    print(archivo.name, '→', resultado.tabla_simbolos)

01_basico_valido.txt → {'x': 'entero', 'limite': 'entero'}
02_expresiones.txt → {'x': 'entero'}
03_control.txt → {'x': 'entero'}
07_token_personalizado.txt → {'numero': 'entero', 'divisor': 'entero', 'residuo': 'entero'}
13_valido_relacionales.txt → {'x': 'entero'}


## 10. Ejemplos inválidos

Se comprueban excepciones de nombres y tipos. No se fabrican cadenas ni decimales
que la gramática no reconoce. Las comparaciones encadenadas sin paréntesis son
error de sintaxis en el parser propio; con paréntesis se pueden probar operandos
booleanos en semántica. Es una diferencia documentada frente a una prueba docente.

In [10]:
for prefijo, clase in [('08_', NameError), ('09_', NameError), ('10_', NameError),
                       ('11_', TypeError), ('12_', TypeError), ('14_', TypeError),
                       ('15_', NameError), ('16_', TypeError)]:
    archivo = next((raiz / 'pruebas').glob(prefijo + '*.txt'))
    print(archivo.name)
    verificar_error(archivo.read_text(encoding='utf-8'), clase)
verificar_error('si (1 < 3 < 5) {}', SyntaxError)
verificar_error('si ((1 < 3) < 5) {}', TypeError)

08_error_variable_no_declarada.txt
NameError [Error Semantico] Línea 1: variable 'x' no declarada.
09_error_variable_duplicada.txt
NameError [Error Semantico] Línea 2: variable 'x' ya declarada.
10_error_identificador_inexistente.txt
NameError [Error Semantico] Línea 1: variable 'fantasma' no declarada.
11_error_tipo_incompatible.txt
TypeError [Error Semantico] Línea 1: inicialización de 'x' exige 'entero', se recibió 'booleano'.
12_error_condicion_no_booleana.txt
TypeError [Error Semantico] Condición de 'si' exige 'booleano', se recibió 'entero'.


14_error_asignacion_tipo.txt
TypeError [Error Semantico] Línea 2: asignación a 'x' exige 'entero', se recibió 'booleano'.
15_error_uso_antes_declarar.txt
NameError [Error Semantico] Línea 1: variable 'x' no declarada.
16_error_modulo_booleano.txt
TypeError [Error Semantico] Operador '%' exige dos enteros, se recibieron 'booleano' y 'entero'.
SyntaxError Línea 1, columna 11: esperado PAR_DER; recibido OP_REL '<'
TypeError [Error Semantico] Operador '<' exige dos enteros, se recibieron 'booleano' y 'entero'.


## 11. Pruebas automatizadas

La suite incluye las siete pruebas originales sin modificaciones y las nuevas
pruebas semánticas. Se muestran los resultados reales y se exige éxito completo.

In [11]:
import io
import unittest
registro = io.StringIO()
suite = unittest.defaultTestLoader.discover(str(raiz / 'pruebas'), pattern='test_*.py')
resultado = unittest.TextTestRunner(stream=registro, verbosity=1).run(suite)
print(registro.getvalue())
assert resultado.wasSuccessful()
assert not resultado.skipped
print('Pruebas:', resultado.testsRun, 'Fallos:', len(resultado.failures),
      'Errores:', len(resultado.errors), 'Omitidas:', len(resultado.skipped))

.........................................
----------------------------------------------------------------------
Ran 41 tests in 0.053s

OK

Pruebas: 41 Fallos: 0 Errores: 0 Omitidas: 0


## 12. Resultados y conclusiones

El recorrido propaga tipos y valida nombres, conserva las extensiones y usa una
tabla global conforme al docente. Las salidas anteriores sustentan los resultados;
el siguiente resumen se calcula desde la ejecución de la suite.

Límites: sin intérprete, TAC, ámbitos locales, análisis de inicialización ni
evaluación de divisores. Solo se conserva el primer error. Para estudiar la
implementación consultar [la guía técnica](../docs/06_GUIA_ANALISIS_SEMANTICO.md)
y [el guion de exposición](../docs/08_GUION_EXPOSICION_SEMANTICA.md).

In [12]:
print(f'Resultado verificado: {resultado.testsRun} pruebas, '
      f'{len(resultado.failures)} fallos y {len(resultado.errors)} errores.')
print('Módulo y tabla global comprobados con los módulos reales del proyecto.')

Resultado verificado: 41 pruebas, 0 fallos y 0 errores.
Módulo y tabla global comprobados con los módulos reales del proyecto.
